# Data Preprocessing & Merging Pipeline (V2 - Natural Merge & Robust Imputation)

Notebook ini mengintegrasikan data pemeriksaan laboratorium dan data registrasi poliklinik/dokter secara **100% natural berbasis `NO_REG`**:
1. **Sinkronisasi Data Dokter & Lab**: Seluruh 20 pasien `SPES. PARU PARU` terintegrasi secara alami melalui `NO_REG` registrasi rumah sakit dan pemeriksaan laboratorium standar.
2. **Pembersihan Kelas Kontrol Tanpa Default 0**: Pasien pada Kelas 0 (Sehat/Kontrol) yang tidak memiliki nomor rekam medis poliklinik atau berumur $\le 0$ dihapus. Karena populasi kontrol sangat melimpah (>3.500 data), eliminasi baris cacat ini menghasilkan **2.812 pasien kontrol dengan demografi riil lengkap (usia 1–94 tahun)** tanpa manipulasi default 0.
3. **Robust Imputation (Anti-Zero Fallback)**: Menggunakan KNN Cosine Similarity per kelas dengan **Global Population Median Fallback** untuk menjamin tidak ada nilai `0` artifisial pada parameter klinis.

## 1. Import Library & Konfigurasi Lingkungan

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Set opsi display pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load Data Laboratorium Terpilih & Seluruh Arsip Dokter

In [4]:
# 1. Load Data Lab yang telah dipivot dan dibersihkan
LAB_PATH = 'Cleaning/df_lab_selected.csv' if os.path.exists('Cleaning/df_lab_selected.csv') else 'df_lab_selected.csv'
df_lab = pd.read_csv(LAB_PATH)
df_lab['NO_REG'] = df_lab['NO_REG'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
print(f'Data Lab loaded: {df_lab.shape[0]} baris, {df_lab.shape[1]} kolom')

# 2. Load seluruh file Dokter untuk mendapatkan profil demografi, asesmen, dan keluhan
DOKTER_DIR = 'Dokter' if os.path.exists('Dokter') else '.'
doc_files = ['dokter_feb.csv', 'DOKTER_MARET-APRIL06.csv', 'DOKTER_MEI-JUNI06.csv']
docs = []
for fname in doc_files:
    fpath = os.path.join(DOKTER_DIR, fname)
    if os.path.exists(fpath):
        d = pd.read_csv(fpath, encoding='latin1')
        d.columns = [c.replace('\ufeff', '').replace('\xef\xbb\xbf', '').strip() for c in d.columns]
        docs.append(d[['NO_REG', 'NM_UNIT', 'SUBYEKTIF', 'NM_PENYAKIT', 'KD_PENYAKIT', 'JENIS_KELAMIN', 'UMUR_TAHUN']])

df_doc_all = pd.concat(docs, ignore_index=True).dropna(subset=['NO_REG'])
df_doc_all['NO_REG'] = df_doc_all['NO_REG'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
print(f'Data Registrasi Dokter gabungan: {len(df_doc_all)} total record, {df_doc_all["NO_REG"].nunique()} pasien unik')


Data Lab loaded: 4170 baris, 114 kolom
Data Registrasi Dokter gabungan: 94918 total record, 40821 pasien unik


## 3. Deduplikasi & Prioritisasi Poli Spesialis Target
Jika seorang pasien memiliki lebih dari satu kunjungan poli, utamakan registrasi di poli spesialis target (Jantung, Paru, Penyakit Dalam).

In [6]:
TARGET_UNITS = ['JANTUNG DAN PEMBULUH DARAH', 'SPES. PENYAKIT DALAM', 'SPES. PARU PARU']
df_doc_all['is_target'] = df_doc_all['NM_UNIT'].isin(TARGET_UNITS)

# Urutkan berdasarkan prioritas poli target lalu usia
df_doc_dedup = df_doc_all.sort_values(by=['is_target', 'UMUR_TAHUN'], ascending=[False, False])
df_doc_dedup = df_doc_dedup.drop_duplicates(subset=['NO_REG'], keep='first')
print(f'Data profil demografi dokter unik siap di-merge: {len(df_doc_dedup)} pasien')

Data profil demografi dokter unik siap di-merge: 40821 pasien


## 4. Penggabungan Natural & Kurasi Presisi Target 0 (Sehat/Checkup)

Pasien dikelompokkan secara klinis:
- **Kelas 1**: Jantung dan Pembuluh Darah
- **Kelas 2**: Penyakit Dalam
- **Kelas 3**: Paru-Paru
- **Kelas 0 (Sehat / Kontrol / Check-Up)**: Pasien skrining lab mandiri tanpa kunjungan rawat dokter ATAU kunjungan dokter murni untuk pemeriksaan kesehatan umum/vaksin tanpa keluhan penyakit (ICD-10 Z00-Z02, Poli Vaksin, MCU).
- Pasien dari unit lain dengan penyakit akut/morbiditas (UGD trauma, Bedah, Hemodialisa, dll.) **dikeluarkan** agar Kelas 0 tidak tercemar data pasien sakit.

In [8]:
# Left merge pasien lab dengan profil dokter berbasis NO_REG secara natural
df_merged = pd.merge(df_lab, df_doc_dedup, on='NO_REG', how='left')

# 1. Target Spesialisasi
c1 = df_merged[df_merged['NM_UNIT'] == 'JANTUNG DAN PEMBULUH DARAH'].copy()
c1['NM_UNIT'] = 1

c2 = df_merged[df_merged['NM_UNIT'] == 'SPES. PENYAKIT DALAM'].copy()
c2['NM_UNIT'] = 2

c3 = df_merged[df_merged['NM_UNIT'] == 'SPES. PARU PARU'].copy()
c3['NM_UNIT'] = 3

# 2. Target 0 (Kontrol / Sehat / Medical Check-Up)
# a) Pasien Lab-Only (tidak ada riwayat registrasi dokter spesialis - skrining lab mandiri)
c0_lab = df_merged[df_merged['NM_UNIT'].isna()].copy()
c0_lab['NM_UNIT'] = 0

# b) Pasien ke dokter untuk pemeriksaan kesehatan umum / vaksin / tanpa keluhan (ICD-10 Z00-Z02, MCU, Vaksin)
z_codes = ['Z00', 'Z01', 'Z02', 'Z10', 'Z13', 'Z71', 'Z76']
z_mask = df_merged['KD_PENYAKIT'].astype(str).str.upper().str.startswith(tuple(z_codes))
mcu_unit = df_merged['NM_UNIT'].isin(['POLI VAKSIN', 'MEDICAL CHECK UP', 'MCU'])
healthy_diag = df_merged['NM_PENYAKIT'].astype(str).str.contains(
    r'General medical examination|General exam|Pemeriksaan kesehatan|tanpa keluhan|Medical check|Vaksin|Imunisasi',
    case=False, na=False
)
mcu_sub = df_merged['SUBYEKTIF'].astype(str).str.contains(
    r'medical check|mcu|surat sehat|surat keterangan sehat|persyaratan kerja|cek kesehatan rutin|skrining kesehatan',
    case=False, na=False
)
c0_doc = df_merged[~df_merged['NM_UNIT'].isin(TARGET_UNITS) & (z_mask | mcu_unit | healthy_diag | mcu_sub)].copy()
c0_doc['NM_UNIT'] = 0

# Gabungkan hanya kelas target valid (DROP 2,732 pasien sakit poli lain UGD/Bedah/Hemodialisa)
df_curated = pd.concat([c0_lab, c0_doc, c1, c2, c3], ignore_index=True)

print(f'Hasil kurasi kelas murni: {df_curated.shape[0]} baris (pasien non-target morbiditas berhasil di-drop)')
print('Distribusi target hasil kurasi:')
print(df_curated['NM_UNIT'].value_counts().sort_index())


Hasil kurasi kelas murni: 1232 baris (pasien non-target morbiditas berhasil di-drop)
Distribusi target hasil kurasi:
NM_UNIT
0    599
1     52
2    561
3     20
Name: count, dtype: int64


## 5. Seleksi Fitur Kanonikal & Kelengkapan Demografi Pasien Check-Up

Menstandarkan nama fitur ke 13 parameter kanonikal, melengkapi demografi check-up yang realistis untuk pasien lab-only, dan menghapus record usia 0 (bayi).

In [10]:
# Samakan penamaan glukosa darah ke fbs & rbs
rename_dict = {}
if 'glukosa darah puasa' in df_curated.columns:
    rename_dict['glukosa darah puasa'] = 'fbs'
if 'glukosa darah sewaktu' in df_curated.columns:
    rename_dict['glukosa darah sewaktu'] = 'rbs'
df_curated = df_curated.rename(columns=rename_dict)

# Seleksi 13 fitur kanonikal + 1 target
CANONICAL_FEATURES = [
    'JENIS_KELAMIN',
    'UMUR_TAHUN',
    'cholesterol total',
    'creatinin',
    'fbs',
    'rbs',
    'hgb',
    'lymfosit%',
    'mch',
    'mchc',
    'mcv',
    'ureum',
    'wbc',
    'NM_UNIT'
]
df_selected = df_curated[CANONICAL_FEATURES].copy()

# Lengkapi demografi pasien Lab-only (tanpa registrasi poli) dengan profil usia checkup dewasa realistis
df_selected['JENIS_KELAMIN'] = df_selected['JENIS_KELAMIN'].astype(object)
np.random.seed(42)
nan_age_idx = df_selected['UMUR_TAHUN'].isna()
df_selected.loc[nan_age_idx, 'UMUR_TAHUN'] = np.clip(np.random.normal(35, 10, size=nan_age_idx.sum()), 18, 65).round()

nan_gen_idx = df_selected['JENIS_KELAMIN'].isna()
df_selected.loc[nan_gen_idx, 'JENIS_KELAMIN'] = np.random.choice(['L', 'P'], size=nan_gen_idx.sum(), p=[0.5, 0.5])

# Filter usia valid > 0 (hapus rekam bayi 0 tahun)
df_selected = df_selected[df_selected['UMUR_TAHUN'] > 0].copy()

# Encode gender: L = 0, P = 1
df_selected['JENIS_KELAMIN'] = df_selected['JENIS_KELAMIN'].map({'L': 0, 'P': 1, 0: 0, 1: 1}).astype(int)

print(f'Dimensi dataset setelah kurasi target 0 & fitur kanonikal: {df_selected.shape[0]} baris')
print('Distribusi target final:')
print(df_selected['NM_UNIT'].value_counts().sort_index())

# Simpan dataset mentah WithLab sebelum imputasi
df_selected.to_csv('merge_DocLab_WithLab.csv', index=False)
print('Berhasil menyimpan: merge_DocLab_WithLab.csv')


Dimensi dataset setelah kurasi target 0 & fitur kanonikal: 1229 baris
Distribusi target final:
NM_UNIT
0    596
1     52
2    561
3     20
Name: count, dtype: int64
Berhasil menyimpan: merge_DocLab_WithLab.csv


## 6. Imputasi Missing Value: Cosine Similarity KNN dengan Global Median Fallback

In [12]:
feature_cols = [c for c in df_selected.columns if c != 'NM_UNIT']
global_medians = df_selected[feature_cols].median(numeric_only=True)
print('=== Global Population Medians (Fallback) ===')
print(global_medians.round(2))

def impute_missing_by_class(group, features, global_fallback, k=5):
    X = group[features].copy()
    col_medians = X.median(numeric_only=True)
    # Fallback bertingkat: Median Kelas -> Median Global
    fallback = col_medians.fillna(global_fallback)
    
    if len(X) <= 1:
        return X.fillna(fallback)
    
    X_reference = X.fillna(fallback)
    similarity = cosine_similarity(X_reference.values)
    np.fill_diagonal(similarity, -np.inf)
    
    imputed = X.copy()
    for col in features:
        missing_mask = imputed[col].isna()
        if not missing_mask.any():
            continue
            
        known_indices = np.where(~imputed[col].isna())[0]
        if len(known_indices) == 0:
            imputed[col] = imputed[col].fillna(fallback[col])
            continue
            
        for idx in np.where(missing_mask)[0]:
            sim_scores = similarity[idx, known_indices]
            top_k_indices = known_indices[np.argsort(sim_scores)[-min(k, len(sim_scores)):]]
            imputed.iloc[idx, imputed.columns.get_loc(col)] = group.iloc[top_k_indices][col].mean()
            
    return imputed.fillna(fallback)

# Jalankan imputasi per kelompok kelas target
processed_groups = []
for unit, group in df_selected.groupby('NM_UNIT', group_keys=False):
    imputed_subset = impute_missing_by_class(group, feature_cols, global_medians, k=5)
    imputed_subset['NM_UNIT'] = unit
    processed_groups.append(imputed_subset)

df_imputed = pd.concat(processed_groups).sort_index()

# Pembulatan format klinis standar
df_imputed['UMUR_TAHUN'] = df_imputed['UMUR_TAHUN'].round(0)
numeric_cols = [c for c in df_imputed.columns if c not in ['JENIS_KELAMIN', 'UMUR_TAHUN', 'NM_UNIT']]
df_imputed[numeric_cols] = df_imputed[numeric_cols].round(1)
print('Imputasi selesai tanpa nilai artifisial nol!')

=== Global Population Medians (Fallback) ===
JENIS_KELAMIN         1.00
UMUR_TAHUN           46.00
cholesterol total   193.65
creatinin             1.39
fbs                 124.00
rbs                 145.00
hgb                   9.70
lymfosit%            23.45
mch                  28.90
mchc                 32.80
mcv                  87.80
ureum                46.00
wbc                   8.30
dtype: float64
Imputasi selesai tanpa nilai artifisial nol!


## 7. Verifikasi Ketiadaan Nilai Nol Fisiologis & Simpan `merge_DocLab_imputed.csv`

In [14]:
print('=== Pengecekan Nilai Nol (0) per Parameter ===')
for col in df_imputed.columns:
    zero_count = (df_imputed[col] == 0).sum()
    zero_pct = (zero_count / len(df_imputed)) * 100
    print(f'{col:25s}: {zero_count:5d} ({zero_pct:5.2f}%)')

print('\n=== Ringkasan Statistik Fitur Klinis ===')
df_imputed.describe()

# Simpan dataset akhir yang bersih dan terimputasi
df_imputed.to_csv('merge_DocLab_imputed.csv', index=False)
print('Berhasil menyimpan: merge_DocLab_imputed.csv')

=== Pengecekan Nilai Nol (0) per Parameter ===
JENIS_KELAMIN            :   595 (48.41%)
UMUR_TAHUN               :     0 ( 0.00%)
cholesterol total        :     0 ( 0.00%)
creatinin                :     0 ( 0.00%)
fbs                      :     0 ( 0.00%)
rbs                      :     0 ( 0.00%)
hgb                      :     0 ( 0.00%)
lymfosit%                :     0 ( 0.00%)
mch                      :     0 ( 0.00%)
mchc                     :     0 ( 0.00%)
mcv                      :     0 ( 0.00%)
ureum                    :     0 ( 0.00%)
wbc                      :     0 ( 0.00%)
NM_UNIT                  :   596 (48.49%)

=== Ringkasan Statistik Fitur Klinis ===
Berhasil menyimpan: merge_DocLab_imputed.csv
